## Iris Random Forest Classifier
End-to-end ML pipeline using SparkML, Feature Store, MLflow model versioning, and Model Serving.

**Catalog:** `ml_training` | **Schema:** `iris_classifier`


Steps:
 - Set up environment
 - Pull in Iris dataset
 - Perform EDA 
 - Preprocessing
 - Feature Engineering
 - Feature Store
 - Train/Test Split
 - Hyperparameter Tuning and Evaluation
 - Test data evaluation
 - MLFlow Experiment Tracking
 - Model Serving

In [0]:
# Configuration
CATALOG = "ml_training"
SCHEMA = "iris_classifier"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Using: {CATALOG}.{SCHEMA}")

In [0]:
%pip install optuna databricks-feature-engineering --quiet

In [0]:
%restart_python

In [0]:
import pandas as pd
from sklearn.datasets import load_iris
from pyspark.sql import SparkSession, DataFrame

def load_iris_as_spark_df(spark: SparkSession) -> DataFrame:
    """Load the Iris dataset and convert it to a Spark DataFrame.

    Uses scikit-learn's bundled copy of the Iris dataset as a convenient
    source, then converts to Spark so the rest of the pipeline stays
    fully within SparkML.

    Args:
        spark: Active SparkSession.

    Returns:
        Iris data with feature columns and a species label column.
    """
    # Load from scikit-learn since SparkML has no built-in datasets
    iris = load_iris()

    # Build a pandas DataFrame with readable column names
    pdf = pd.DataFrame(iris.data, columns=[
        "sepal_length", "sepal_width", "petal_length", "petal_width"
    ])

    # Map numeric targets to species names for interpretability
    species_map = dict(enumerate(iris.target_names))
    pdf["species"] = pd.Series(iris.target).map(species_map)

    # Add a unique ID column -- required later for Feature Store lookups
    pdf["iris_id"] = range(len(pdf))

    return spark.createDataFrame(pdf)


iris_df = load_iris_as_spark_df(spark)
display(iris_df)

In [0]:
import numpy as np
import pandas as pd

# SparkML requires assembling features into a single vector column
from pyspark.ml.feature import VectorAssembler, StringIndexer

# Core model and pipeline components
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# SparkML uses a randomSplit method on DataFrames rather than a standalone
# train_test_split function like scikit-learn
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Optuna is used for hyperparameter tuning
import optuna

print("Dependencies loaded.")

In [0]:
# EDA: Count of each species
species_count_df = iris_df.groupBy("species").count()
display(species_count_df)

# EDA: Boxplots for each feature grouped by species
# Spark DataFrames do not support boxplots natively, so convert to pandas for visualization
iris_pd = iris_df.toPandas()

import matplotlib.pyplot as plt
import seaborn as sns

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

for feature in features:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x="species", y=feature, data=iris_pd)
    plt.title(f"Boxplot of {feature} by Species")
    plt.xlabel("Species")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

In [0]:
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.iris_features"

# Separate features from the label. The Feature Store holds only
# the input features keyed by a primary key. The label (species)
# stays in the training DataFrame and gets joined at training time
# via FeatureLookup.
feature_columns = ["iris_id", "sepal_length", "sepal_width", "petal_length", "petal_width"]
features_df = iris_df.select(feature_columns)

# Write features as a managed Delta table. Any Delta table in Unity
# Catalog with a primary key constraint is automatically recognized
# as a feature table -- no special API needed.
features_df.write.format("delta").mode("overwrite").saveAsTable(FEATURE_TABLE)

# Adding a primary key constraint is what makes Unity Catalog treat
# this as a feature table. It enables FeatureLookup joins during
# training and lineage tracking.
spark.sql(f"""
    ALTER TABLE {FEATURE_TABLE}
    ALTER COLUMN iris_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {FEATURE_TABLE}
    ADD CONSTRAINT iris_features_pk PRIMARY KEY (iris_id)
""")

print(f"Feature table created: {FEATURE_TABLE}")
display(spark.table(FEATURE_TABLE))

In [0]:
from pyspark.sql import DataFrame

def stratified_split(
    df: DataFrame,
    label_col: str,
    train_fraction: float = 0.8,
    seed: int = 42
) -> tuple[DataFrame, DataFrame]:
    """Split a DataFrame into train/test sets preserving class proportions.

    Uses Spark's sampleBy to draw a stratified sample for the training
    set, then subtracts it from the original to produce the test set.
    This guarantees each class is represented proportionally in both
    splits, which matters for imbalanced datasets.

    Args:
        df: Input Spark DataFrame.
        label_col: Name of the column containing class labels.
        train_fraction: Fraction of each class to include in training.
        seed: Random seed for reproducibility.

    Returns:
        A tuple of (train_df, test_df) with stratified class distributions.
    """
    # Build a fractions dict with the same ratio for every class so
    # each label is sampled proportionally
    labels = [row[label_col] for row in df.select(label_col).distinct().collect()]
    fractions = {label: train_fraction for label in labels}

    # sampleBy draws approximately train_fraction of each class
    train_df = df.sampleBy(label_col, fractions, seed=seed)

    # subtract removes all training rows, leaving the test set
    test_df = df.subtract(train_df)

    return train_df, test_df


train_df, test_df = stratified_split(iris_df, label_col="species")

print(f"Training set: {train_df.count()} rows")
print(f"Test set:     {test_df.count()} rows")

print("\nTraining class distribution:")
display(train_df.groupBy("species").count().orderBy("species"))

print("\nTest class distribution:")
display(test_df.groupBy("species").count().orderBy("species"))

In [0]:
import mlflow

# Set the experiment before any training runs so all trials are
# grouped together in the MLflow UI for easy comparison
EXPERIMENT_NAME = "/Users/eclark772@gmail.com/iris_classifier/iris_rf_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

# Note: mlflow.autolog() does not work on serverless compute (Spark Connect),
# so we log manually in the Optuna objective. This gives us finer control
# over exactly what gets tracked for each trial.

print(f"MLflow experiment: {EXPERIMENT_NAME}")

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import optuna

FEATURE_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]


def build_pipeline(
    num_trees: int,
    max_depth: int,
    min_instances_per_node: int,
    feature_subset_strategy: str,
    seed: int = 42
) -> Pipeline:
    """Build a SparkML pipeline for Iris classification.

    Constructs a three-stage pipeline:
      1. StringIndexer: converts the string species label to a numeric index
      2. VectorAssembler: combines individual feature columns into a single
         vector column, which is the format SparkML models expect
      3. RandomForestClassifier: the model itself, parameterized by the
         hyperparameters passed in

    Args:
        num_trees: Number of trees in the forest.
        max_depth: Maximum depth of each tree.
        min_instances_per_node: Minimum samples required at a leaf node.
        feature_subset_strategy: How many features to consider per split
            (e.g., "auto", "sqrt", "log2").
        seed: Random seed for reproducibility.

    Returns:
        An unfitted SparkML Pipeline ready for .fit().
    """
    # StringIndexer learns the mapping from species names to numeric
    # indices (e.g., setosa=0, versicolor=1, virginica=2)
    label_indexer = StringIndexer(
        inputCol="species",
        outputCol="label",
        handleInvalid="keep"
    )

    # VectorAssembler is required because SparkML models expect a single
    # vector column as input rather than separate feature columns
    assembler = VectorAssembler(
        inputCols=FEATURE_COLS,
        outputCol="features"
    )

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=num_trees,
        maxDepth=max_depth,
        minInstancesPerNode=min_instances_per_node,
        featureSubsetStrategy=feature_subset_strategy,
        seed=seed
    )

    return Pipeline(stages=[label_indexer, assembler, rf])


def objective(trial: optuna.Trial) -> float:
    """Optuna objective function for hyperparameter tuning.

    Each trial suggests a set of hyperparameters, builds and fits the
    pipeline, evaluates it, and logs everything to MLflow. Optuna's
    TPE sampler uses results from prior trials to focus the search
    on the most promising regions of the hyperparameter space.

    Args:
        trial: Optuna trial object that suggests hyperparameter values.

    Returns:
        The weighted F1 score on the test set (higher is better).
    """
    # suggest_int/suggest_categorical define the search space.
    # Optuna decides which values to try based on prior trial results.
    num_trees = trial.suggest_int("num_trees", 10, 200, step=10)
    max_depth = trial.suggest_int("max_depth", 2, 15)
    min_instances_per_node = trial.suggest_int("min_instances_per_node", 1, 10)
    feature_subset_strategy = trial.suggest_categorical(
        "feature_subset_strategy", ["auto", "sqrt", "log2"]
    )

    pipeline = build_pipeline(
        num_trees=num_trees,
        max_depth=max_depth,
        min_instances_per_node=min_instances_per_node,
        feature_subset_strategy=feature_subset_strategy
    )

    # Fit the full pipeline on training data -- StringIndexer and
    # VectorAssembler transformations are learned here too
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    # Evaluate using multiple metrics to get a complete picture
    evaluator = MulticlassClassificationEvaluator(labelCol="label")

    f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
    accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
    precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
    recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

    # Log each trial as a nested run under the parent so the MLflow UI
    # shows a clean hierarchy: one parent run with N child trials
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params({
            "num_trees": num_trees,
            "max_depth": max_depth,
            "min_instances_per_node": min_instances_per_node,
            "feature_subset_strategy": feature_subset_strategy
        })
        mlflow.log_metrics({
            "f1": f1,
            "accuracy": accuracy,
            "weighted_precision": precision,
            "weighted_recall": recall
        })

    return f1


print("Pipeline and objective function defined.")

In [0]:
N_TRIALS = 20

# Wrap the entire study in a parent MLflow run so the experiment UI
# shows one top-level run containing all trial child runs
with mlflow.start_run(run_name="optuna_rf_tuning") as parent_run:

    # create_study with "maximize" because we want the highest F1 score.
    # TPE (default sampler) is the Bayesian optimizer that learns from
    # previous trials to suggest better hyperparameters.
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    # Log the best trial's params and score to the parent run for
    # quick reference without drilling into child runs
    best = study.best_trial
    mlflow.log_params({f"best_{k}": v for k, v in best.params.items()})
    mlflow.log_metric("best_f1", best.value)

    print(f"Best F1 score: {best.value:.4f}")
    print(f"Best parameters: {best.params}")
    print(f"Parent run ID: {parent_run.info.run_id}")

In [0]:
from pyspark.ml import PipelineModel
from pyspark.sql import DataFrame
import matplotlib.pyplot as plt
import numpy as np

def evaluate_best_model(
    study: optuna.Study,
    train_df: DataFrame,
    test_df: DataFrame
) -> tuple[PipelineModel, DataFrame]:
    """Retrain the best model from the Optuna study and evaluate on test data.

    Rebuilds the pipeline with the best hyperparameters found during
    tuning, fits on training data, and produces predictions on the
    held-out test set. This gives us an honest estimate of generalization
    performance since the test set was never used during tuning.

    Args:
        study: Completed Optuna study containing trial results.
        train_df: Training Spark DataFrame.
        test_df: Test Spark DataFrame.

    Returns:
        A tuple of (fitted PipelineModel, predictions DataFrame).
    """
    best_params = study.best_trial.params

    # Rebuild pipeline with the winning hyperparameters
    best_pipeline = build_pipeline(
        num_trees=best_params["num_trees"],
        max_depth=best_params["max_depth"],
        min_instances_per_node=best_params["min_instances_per_node"],
        feature_subset_strategy=best_params["feature_subset_strategy"]
    )

    best_model = best_pipeline.fit(train_df)
    predictions = best_model.transform(test_df)

    return best_model, predictions


# Retrain and predict with the best hyperparameters
best_model, predictions = evaluate_best_model(study, train_df, test_df)

# Compute all relevant classification metrics on the test set
evaluator = MulticlassClassificationEvaluator(labelCol="label")

metrics = {
    "test_f1": evaluator.evaluate(predictions, {evaluator.metricName: "f1"}),
    "test_accuracy": evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"}),
    "test_weighted_precision": evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"}),
    "test_weighted_recall": evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"}),
}

print("Test Set Evaluation Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

# Show predictions alongside actual labels for inspection
print("\nSample predictions:")
display(
    predictions.select("iris_id", "species", "label", "prediction", "probability")
)

In [0]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(
    predictions: DataFrame,
    label_col: str = "label",
    prediction_col: str = "prediction"
) -> plt.Figure:
    """Plot a confusion matrix from SparkML predictions.

    Converts Spark predictions to pandas for visualization since
    matplotlib operates on local data. The confusion matrix shows
    where the model confuses one class for another, which is more
    informative than a single accuracy number.

    Args:
        predictions: Spark DataFrame with label and prediction columns.
        label_col: Name of the true label column.
        prediction_col: Name of the predicted label column.

    Returns:
        The matplotlib Figure containing the confusion matrix.
    """
    pred_pdf = predictions.select(label_col, prediction_col).toPandas()

    # Retrieve the species names from the StringIndexer so the matrix
    # labels show readable names instead of numeric indices
    label_indexer_model = best_model.stages[0]
    class_labels = label_indexer_model.labels

    cm = confusion_matrix(
        pred_pdf[label_col],
        pred_pdf[prediction_col]
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("Confusion Matrix -- Test Set")
    plt.tight_layout()
    plt.show()

    return fig


# Log final metrics and confusion matrix to MLflow under a dedicated
# evaluation run so it is easy to find alongside the tuning runs
with mlflow.start_run(run_name="best_model_evaluation"):
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics(metrics)

    fig = plot_confusion_matrix(predictions)
    mlflow.log_figure(fig, "confusion_matrix.png")

    print("Metrics and confusion matrix logged to MLflow.")

In [0]:
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.preprocessing import LabelEncoder
import mlflow.pyfunc

# Registering in ws_sandbox since the MLflow Model Registry API
# requires different UC privileges than Spark SQL. The ml_training
# catalog allows table creation via SQL but not API-level model
# registration -- a workspace admin can grant this later.
MODEL_NAME = "ws_sandbox.default.iris_rf_model"

mlflow.set_registry_uri("databricks-uc")


def map_feature_strategy(strategy: str) -> str | None:
    """Map SparkML featureSubsetStrategy to sklearn max_features.

    SparkML uses "auto" to mean "sqrt" for classification, but
    newer sklearn versions removed "auto" as a valid option.

    Args:
        strategy: SparkML feature subset strategy name.

    Returns:
        The equivalent sklearn max_features value.
    """
    mapping = {"auto": "sqrt", "sqrt": "sqrt", "log2": "log2", "all": None}
    return mapping.get(strategy, "sqrt")


class IrisRFWrapper(mlflow.pyfunc.PythonModel):
    """Custom pyfunc wrapper for the Iris Random Forest classifier.

    Wraps a scikit-learn RF model trained with the same best
    hyperparameters found by Optuna. We use pyfunc instead of
    mlflow.spark.log_model because serving endpoints don't run
    Spark -- a lightweight sklearn model gives faster inference
    and simpler deployment.

    The model accepts raw feature columns as input and returns
    the predicted species name.
    """

    def __init__(
        self,
        sklearn_model: SklearnRF,
        label_encoder: LabelEncoder,
        feature_cols: list[str]
    ) -> None:
        """Initialize the wrapper with a trained model and label mapping.

        Args:
            sklearn_model: Fitted sklearn RandomForestClassifier.
            label_encoder: Fitted LabelEncoder for species names.
            feature_cols: Ordered list of input feature column names.
        """
        self.sklearn_model = sklearn_model
        self.label_encoder = label_encoder
        self.feature_cols = feature_cols

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """Generate predictions from raw feature input.

        Args:
            context: MLflow context (unused but required by interface).
            model_input: DataFrame with the four Iris feature columns.

        Returns:
            DataFrame with predicted species name.
        """
        X = model_input[self.feature_cols]
        pred_indices = self.sklearn_model.predict(X)
        pred_labels = self.label_encoder.inverse_transform(pred_indices)

        return pd.DataFrame({"predicted_species": pred_labels})


# Convert Spark DataFrames to pandas for sklearn training
train_pdf = train_df.toPandas()
test_pdf = test_df.toPandas()

# Encode species labels to match the SparkML StringIndexer behavior
le = LabelEncoder()
le.fit(train_pdf["species"])
y_train = le.transform(train_pdf["species"])
X_train = train_pdf[FEATURE_COLS]

# Train sklearn RF with the same hyperparameters Optuna found best.
# This produces an equivalent model to the SparkML version but
# serializable without Spark -- ideal for serving.
best_params = study.best_trial.params
sklearn_rf = SklearnRF(
    n_estimators=best_params["num_trees"],
    max_depth=best_params["max_depth"],
    min_samples_leaf=best_params["min_instances_per_node"],
    max_features=map_feature_strategy(best_params["feature_subset_strategy"]),
    random_state=42,
    n_jobs=-1
)
sklearn_rf.fit(X_train, y_train)

# Build the pyfunc wrapper
wrapped_model = IrisRFWrapper(
    sklearn_model=sklearn_rf,
    label_encoder=le,
    feature_cols=FEATURE_COLS
)

# Create signature and input_example -- both required for UC
# registration and serving endpoint compatibility
input_sample = test_pdf[FEATURE_COLS].head(5)
output_sample = wrapped_model.predict(None, input_sample)
signature = infer_signature(input_sample, output_sample)

# Log the model and register in Unity Catalog
with mlflow.start_run(run_name="best_model_registration") as run:
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics(metrics)

    model_info = mlflow.pyfunc.log_model(
        artifact_path="iris-rf-model",
        python_model=wrapped_model,
        signature=signature,
        input_example=input_sample,
        registered_model_name=MODEL_NAME,
        pip_requirements=["scikit-learn", "pandas", "numpy"]
    )

    print(f"Model registered: {MODEL_NAME}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

ENDPOINT_NAME = "iris-rf-classifier"

# Create a serving endpoint that hosts our registered model.
# Serverless serving is the simplest option -- Databricks manages
# the infrastructure and scales to zero when idle.
endpoint = w.serving_endpoints.create_and_wait(
    name=ENDPOINT_NAME,
    config=EndpointCoreConfigInput(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                name="iris-rf-entity",
                entity_name=MODEL_NAME,
                entity_version="1",
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    ),
)

print(f"Serving endpoint created: {ENDPOINT_NAME}")
print(f"State: {endpoint.state.ready}")

In [0]:
import json

# Send a test request to the serving endpoint to verify it works
# end-to-end. This simulates what an application would send.
test_data = {
    "dataframe_records": [
        {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
        {"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.2, "petal_width": 2.3},
        {"sepal_length": 5.9, "sepal_width": 3.0, "petal_length": 4.2, "petal_width": 1.5},
    ]
}

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=test_data["dataframe_records"]
)

print("Serving endpoint test results:")
print(json.dumps(response.as_dict(), indent=2))

In [0]:
def run_inference_test(
    endpoint_name: str,
    test_pdf: pd.DataFrame,
    feature_cols: list[str],
    n_samples: int = 10
) -> pd.DataFrame:
    """Send test samples to the serving endpoint and compare against actuals.

    Simulates a real-world inference workflow: select samples, send
    them as JSON to the REST endpoint, then compare returned predictions
    against the known ground truth labels.

    Args:
        endpoint_name: Name of the deployed serving endpoint.
        test_pdf: Pandas DataFrame containing test data with labels.
        feature_cols: List of feature column names to send.
        n_samples: Number of random samples to test.

    Returns:
        DataFrame with features, actual species, predicted species,
        and a match indicator.
    """
    # Sample from the test set so we have ground truth to compare against
    samples = test_pdf.sample(n=min(n_samples, len(test_pdf)), random_state=42)

    # Build the request payload using only the feature columns --
    # the endpoint should never see the label
    records = samples[feature_cols].to_dict(orient="records")

    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_records=records
    )

    # Combine predictions with ground truth for comparison
    predictions = [p["predicted_species"] for p in response.predictions]
    results = samples[feature_cols + ["species"]].copy()
    results["predicted"] = predictions
    results["correct"] = results["species"] == results["predicted"]

    return results.reset_index(drop=True)


results = run_inference_test(ENDPOINT_NAME, test_pdf, FEATURE_COLS)

accuracy = results["correct"].mean()
print(f"Inference test accuracy: {accuracy:.0%} ({results['correct'].sum()}/{len(results)})\n")
display(results)